# Mixed-Precision 양자화 (로컬 버전)

## 개요
레이어별로 다른 bit-width를 적용하여 성능과 압축률의 균형을 최적화합니다.

### 실행 전 필수 사항
```bash
cd lg-aimers8-llm-compression
source venv/bin/activate
jupyter notebook
```

---

# 1. Import 및 환경 확인

In [ ]:
import os
import sys
import torch
import shutil
from pathlib import Path

from datasets import load_dataset
from transformers import AutoModelForCausalLM, AutoTokenizer

from llmcompressor import oneshot
from llmcompressor.modifiers.quantization import GPTQModifier

print("=" * 60)
print("환경 정보")
print("=" * 60)
print(f"Python: {sys.version.split()[0]}")
print(f"PyTorch: {torch.__version__}")
print(f"CUDA 사용 가능: {torch.cuda.is_available()}")
if torch.cuda.is_available():
    print(f"GPU: {torch.cuda.get_device_name(0)}")
else:
    print("⚠️  GPU 없음 - CPU로 실행됩니다")
print("=" * 60)

# 2. 하이퍼파라미터 설정

In [ ]:
# ============================================================================
# 모델 설정
# ============================================================================
# 로컬 모델 경로 (다운로드 불필요!)
MODEL_ID = "./open/base_model"
OUT_DIR = "./model_mixed_precision"

# 데이터셋 설정
DATASET_ID = "LGAI-EXAONE/MANTA-1M"
DATASET_SPLIT = "train"

# 캘리브레이션 설정
if torch.cuda.is_available():
    NUM_CALIBRATION_SAMPLES = 512
    MAX_SEQUENCE_LENGTH = 1024
else:
    NUM_CALIBRATION_SAMPLES = 256
    MAX_SEQUENCE_LENGTH = 512

# ============================================================================
# EXAONE-4.0-1.2B 모델 구조: 총 30개 레이어
# ============================================================================
NUM_LAYERS = 30

# ============================================================================
# Mixed-Precision 전략
# 민감 레이어 (FP16 유지): 첫/마지막 레이어
# 나머지 레이어: 4-bit 양자화
# ============================================================================
SENSITIVE_LAYERS = [
    "model.layers.0",
    "model.layers.1",
    "model.layers.28",
    "model.layers.29",
]

ALWAYS_IGNORE = ["embed_tokens", "lm_head"]

# 원본 모델 크기
ORIGINAL_MODEL_SIZE_GB = 2.56

print("=" * 60)
print("Mixed-Precision 설정")
print("=" * 60)
print(f"MODEL_ID: {MODEL_ID}")
print(f"총 레이어 수: {NUM_LAYERS}")
print(f"민감 레이어 (FP16): {len(SENSITIVE_LAYERS)}개")
print(f"양자화 레이어 (4-bit): {NUM_LAYERS - len(SENSITIVE_LAYERS)}개")
print("=" * 60)

# 3. 모델 로드

In [ ]:
print("[INFO] 모델 로드 중...")

tokenizer = AutoTokenizer.from_pretrained(
    MODEL_ID,
    trust_remote_code=True,
)

device_map = "auto" if torch.cuda.is_available() else "cpu"

model = AutoModelForCausalLM.from_pretrained(
    MODEL_ID,
    torch_dtype=torch.float32 if not torch.cuda.is_available() else torch.bfloat16,
    trust_remote_code=True,
    device_map=device_map,
)

print(f"[INFO] 모델 파라미터: {model.num_parameters():,}")
print(f"[INFO] 디바이스: {device_map}")
print("[INFO] 모델 로드 완료")

# 4. 데이터셋 로드

In [ ]:
print("[INFO] 캘리브레이션 데이터 로드 중...")

ds = load_dataset(
    DATASET_ID,
    split=f"{DATASET_SPLIT}[:{NUM_CALIBRATION_SAMPLES}]",
)

def preprocess(example):
    return {
        "text": tokenizer.apply_chat_template(
            example["conversations"],
            add_generation_prompt=True,
            tokenize=False
        )
    }

ds = ds.map(preprocess)

print(f"[INFO] 데이터셋 크기: {len(ds)}")

# 5. Mixed-Precision 양자화

In [ ]:
print("[INFO] Mixed-Precision 양자화 시작")
print(f"  - 민감 레이어 (FP16 유지): {SENSITIVE_LAYERS}")
print(f"  - 나머지 레이어: W4A16")

if torch.cuda.is_available():
    print("\n🚀 GPU 모드: 15-30분 예상\n")
else:
    print("\n⏳ CPU 모드: 2-5시간 예상\n")

# 제외할 레이어 목록
ignore_layers = ALWAYS_IGNORE + SENSITIVE_LAYERS

recipe = [
    GPTQModifier(
        scheme="W4A16",
        targets=["Linear"],
        ignore=ignore_layers,
        block_size=128,
    )
]

oneshot(
    model=model,
    dataset=ds,
    recipe=recipe,
    max_seq_length=MAX_SEQUENCE_LENGTH,
    num_calibration_samples=NUM_CALIBRATION_SAMPLES,
)

print("\n[INFO] Mixed-Precision 양자화 완료!")

# 6. 모델 저장 및 크기 비교

In [ ]:
print("[INFO] 모델 저장 중...")

os.makedirs(OUT_DIR, exist_ok=True)
model.save_pretrained(OUT_DIR, save_compressed=True)
tokenizer.save_pretrained(OUT_DIR)

# 파일 확인
print(f"\n[INFO] 저장된 파일:")
total_size = 0
for f in sorted(os.listdir(OUT_DIR)):
    size = os.path.getsize(os.path.join(OUT_DIR, f))
    total_size += size
    print(f"  {f}: {size/1e6:.1f} MB")

quantized_size_gb = total_size / 1e9

print("\n" + "=" * 60)
print("모델 크기 비교")
print("=" * 60)
print(f"  원본 모델:     {ORIGINAL_MODEL_SIZE_GB:.2f} GB")
print(f"  양자화 모델:   {quantized_size_gb:.2f} GB")
print(f"  ----------------------------------------")
print(f"  크기 감소:     {ORIGINAL_MODEL_SIZE_GB - quantized_size_gb:.2f} GB")
print(f"  압축률:        {quantized_size_gb / ORIGINAL_MODEL_SIZE_GB * 100:.1f}%")
print(f"  압축 배수:     {ORIGINAL_MODEL_SIZE_GB / quantized_size_gb:.2f}x")
print("=" * 60)
print(f"\n  민감 레이어 {len(SENSITIVE_LAYERS)}개 FP16 유지")
print(f"  나머지 {NUM_LAYERS - len(SENSITIVE_LAYERS)}개 레이어 4-bit 양자화")

# 7. 제출 파일 생성

In [ ]:
zip_name = "submit_mixed_precision"
print(f"[INFO] {zip_name}.zip 생성 중...")

shutil.make_archive(
    base_name=zip_name,
    format="zip",
    root_dir=".",
    base_dir=OUT_DIR,
)

zip_size = os.path.getsize(f"{zip_name}.zip") / 1e9
print(f"[INFO] 생성 완료: {zip_name}.zip ({zip_size:.2f} GB)")

if zip_size <= 10:
    print("✅ 용량 제한 충족 (≤ 10GB)")
else:
    print("❌ 용량 초과!")

print(f"\n📁 파일 위치: {os.path.abspath(f'{zip_name}.zip')}")

# 8. 모델 테스트

In [ ]:
print("[INFO] Mixed-Precision 모델 테스트...")

test_messages = [
    [{"role": "user", "content": "안녕하세요!"}],
    [{"role": "user", "content": "2 + 3 = ?"}],
]

for i, message in enumerate(test_messages):
    print(f"\n--- 테스트 {i+1} ---")
    print(f"질문: {message[0]['content']}")
    
    input_ids = tokenizer.apply_chat_template(
        message,
        add_generation_prompt=True,
        return_tensors="pt"
    ).to(model.device)
    
    output = model.generate(
        input_ids,
        max_new_tokens=50,
        do_sample=False,
    )
    
    response = tokenizer.decode(output[0], skip_special_tokens=True)
    print(f"응답: {response.split('assistant')[-1].strip() if 'assistant' in response else response}")